In [ ]:
# ============================================================
# Cell 1: Colab Setup + Google Drive Mount
# ============================================================
!pip install -q lightgbm optuna shap pyarrow matplotlib joblib

import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/BIRL_EnvModel'
os.makedirs(DRIVE_DIR, exist_ok=True)
WORK_DIR = '/content'
os.chdir(WORK_DIR)

# Copy data from Drive to local (faster I/O)
for fname in ['birl_env.parquet', 'action_space_config.json']:
    src = os.path.join(DRIVE_DIR, fname)
    dst = os.path.join(WORK_DIR, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'Copied {fname} from Drive')
    elif os.path.exists(dst):
        print(f'{fname} already local')
    else:
        print(f'WARNING: {fname} not found! Upload to {DRIVE_DIR}/')

print(f'\nDrive: {DRIVE_DIR}')
print(f'Drive contents: {[f for f in os.listdir(DRIVE_DIR) if not f.startswith(".")]}')

In [ ]:
# ============================================================
# Cell 2: Imports + Checkpoint Helpers
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import sys, json, time, pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from scipy.stats import norm
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
optuna.logging.set_verbosity(optuna.logging.WARNING)

def save_drive(local_path):
    """Save local file to Drive."""
    dst = os.path.join(DRIVE_DIR, os.path.basename(local_path))
    shutil.copy2(local_path, dst)
    print(f'  -> Drive: {os.path.basename(local_path)}')

def load_drive(filename):
    """Load file from Drive to local. Returns True if found."""
    src = os.path.join(DRIVE_DIR, filename)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(WORK_DIR, filename))
        print(f'  <- Drive: {filename}')
        return True
    return False

def on_drive(name):
    return os.path.exists(os.path.join(DRIVE_DIR, name))

print(f'LightGBM {lgb.__version__}, Optuna {optuna.__version__}')

In [ ]:
# ============================================================
# Cell 3: Load Data
# ============================================================
df = pd.read_parquet('birl_env.parquet')
with open('action_space_config.json') as f:
    action_config = json.load(f)

N_actions = action_config['n_actions']
crop_order = action_config['crop_order']
idx_to_crop = {i: c for i, c in enumerate(crop_order)}

y = np.log(df['harvest_value_USD_w'].clip(lower=0.01) + 1).values
print(f'Data: {df.shape}, Actions: {N_actions}')
print(f'y: mean={y.mean():.3f}, std={y.std():.3f}')

In [ ]:
# ============================================================
# Cell 4: Feature Engineering
# ============================================================
action_features = ['action_crop', 'input_intensity']
state_features = [
    'plot_area_GPS','intercropped','plot_owned','irrigated',
    'rainfall_growing_sum_final','rainfall_10yr_mean_final','rainfall_10yr_cv_final',
    'ndvi_preseason_mean_final','ndvi_growing_mean_final',
    'era5_tmean_growing_final','era5_tmax_growing_final',
    'clay_0-5cm','sand_0-5cm','soc_0-5cm','nitrogen_0-5cm','phh2o_0-5cm',
    'elevation_m','slope_deg',
    'hh_size','hh_asset_index','hh_dependency_ratio',
    'age_manager','female_manager','formal_education_manager',
    'livestock','nonfarm_enterprise','hh_electricity_access',
    'travel_time_city_min','urban',
    'conflict_events_25km_12m','conflict_nearest_event_km',
    'country','year','season',
]
feature_cols = [c for c in action_features + state_features if c in df.columns]
cat_features = ['action_crop', 'country', 'season']

if df['input_intensity'].dtype == object:
    df['input_intensity'] = df['input_intensity'].map({'low':0,'medium':1,'high':2}).astype(float)
for c in cat_features:
    df[c] = df[c].astype('category')

X = df[feature_cols]
print(f'Features: {len(feature_cols)}')
print(f'Confirmed dea_efficiency excluded: {"dea_efficiency" not in feature_cols}')

In [ ]:
# ============================================================
# Cell 5: Train/Test Split
# ============================================================
rng = np.random.default_rng(42)
unique_hh = df['hh_id_merge'].unique()
rng.shuffle(unique_hh)
n_test = int(len(unique_hh) * 0.15)
test_hh = set(unique_hh[:n_test])
trainval_mask = ~df['hh_id_merge'].isin(test_hh)
test_mask = ~trainval_mask

X_trainval, y_trainval = X[trainval_mask], y[trainval_mask]
X_test, y_test = X[test_mask], y[test_mask]
groups_trainval = df.loc[trainval_mask, 'hh_id_merge']

gkf = GroupKFold(n_splits=3)
folds = list(gkf.split(X_trainval, y_trainval, groups=groups_trainval))
dtrain = lgb.Dataset(X_trainval, y_trainval,
                     categorical_feature=cat_features, free_raw_data=False)

print(f'TrainVal: {len(X_trainval):,}, Test: {len(X_test):,}')

In [ ]:
import os, psutil

print(f"CPU cores: {os.cpu_count()}")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
!lscpu | grep "CPU(s):"

---
## Model A: Mean Model (μ)

Optuna with Drive checkpoint: resumes if Colab disconnects.

In [ ]:
# ============================================================
# Cell 6: Model A Optuna (with Drive checkpoint)
# ============================================================
base_params_mu = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbose': -1,
    'n_jobs': -1,
    'feature_pre_filter': False,
    'seed': 42,
}
TARGET_TRIALS_MU = 20

def objective_mu(trial):
    params = base_params_mu | {
        'num_leaves': trial.suggest_int('num_leaves', 63, 255),
        'learning_rate': trial.suggest_float('lr', 0.03, 0.1, log=True),
        'min_child_samples': trial.suggest_int('min_child', 20, 100, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
    }
    cv = lgb.cv(params, dtrain, num_boost_round=4000, folds=folds,
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    return cv['valid rmse-mean'][-1]

# Resume or start fresh
if on_drive('study_mu.pkl'):
    load_drive('study_mu.pkl')
    with open('study_mu.pkl', 'rb') as f: study_mu = pickle.load(f)
    done = len(study_mu.trials)
    print(f'Resumed study_mu: {done}/{TARGET_TRIALS_MU} trials, best={study_mu.best_value:.4f}')
else:
    study_mu = optuna.create_study(
        direction='minimize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    )
    done = 0
    print('Starting fresh study_mu')

remaining = max(0, TARGET_TRIALS_MU - done)
if remaining > 0:
    print(f'Running {remaining} trials...'); sys.stdout.flush()
    t0 = time.time()
    study_mu.optimize(objective_mu, n_trials=remaining, show_progress_bar=True)
    print(f'Done in {(time.time()-t0)/60:.1f} min')
    with open('study_mu.pkl', 'wb') as f: pickle.dump(study_mu, f)
    save_drive('study_mu.pkl')
else:
    print('All trials complete, loaded from Drive')

print(f'Best RMSE={study_mu.best_value:.4f}, params={study_mu.best_params}')

In [ ]:
# ============================================================
# Cell 7: Train Final Model A (or load from Drive)
# ============================================================
if on_drive('model_mu.txt'):
    load_drive('model_mu.txt')
    model_mu = lgb.Booster(model_file='model_mu.txt')
    print('Loaded model_mu from Drive')
else:
    # Map Optuna short names → LightGBM official names
    bp = study_mu.best_params
    best_p = base_params_mu | {
        'num_leaves':        bp['num_leaves'],
        'learning_rate':     bp['lr'],
        'min_child_samples': bp['min_child'],
        'subsample':         bp['subsample'],
        'colsample_bytree':  bp['colsample'],
        'reg_alpha':         bp['reg_alpha'],
        'reg_lambda':        bp['reg_lambda'],
    }
    print(f'Best params: {best_p}')

    cv_f = lgb.cv(best_p, dtrain, num_boost_round=6000, folds=folds,
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    best_round_mu = len(cv_f['valid rmse-mean'])
    print(f'CV best round: {best_round_mu}, CV RMSE: {cv_f["valid rmse-mean"][-1]:.6f}')

    model_mu = lgb.train(best_p, dtrain, num_boost_round=best_round_mu)
    model_mu.save_model('model_mu.txt')
    save_drive('model_mu.txt')
    print(f'Trained model_mu: {best_round_mu} rounds')

# ── OOS evaluation ──
mu_pred_test = model_mu.predict(X_test)
r2 = r2_score(y_test, mu_pred_test)
rmse = np.sqrt(np.mean((y_test - mu_pred_test)**2))
print(f'Model A OOS R2={r2:.4f}, RMSE={rmse:.4f}')

# ── Sanity check: R2 should be 0.30-0.50 for yield, could be higher for harvest value ──
if r2 > 0.95:
    print('⚠️  R2 suspiciously high — check for data leakage or dominant feature (plot_area?)')
    # Quick feature importance check
    imp = model_mu.feature_importance(importance_type='gain')
    feat_names = model_mu.feature_name()
    top5 = sorted(zip(feat_names, imp), key=lambda x: -x[1])[:5]
    print('Top 5 features by gain:')
    for name, gain in top5:
        print(f'  {name}: {gain:.0f} ({gain/imp.sum()*100:.1f}%)')

---
## Model B: Volatility Model (σ)

In [ ]:
# ============================================================
# Cell 8: Model B Optuna (with Drive checkpoint)
# ============================================================
mu_pred_tv = model_mu.predict(X_trainval)
y_sigma = np.abs(y_trainval - mu_pred_tv)
dtrain_sigma = lgb.Dataset(X_trainval, y_sigma,
                           categorical_feature=cat_features, free_raw_data=False)

base_params_sigma = {
    'objective': 'regression',
    'metric': 'mae',
    'verbose': -1,
    'n_jobs': -1,
    'feature_pre_filter': False,
    'seed': 42,
}
TARGET_TRIALS_SIGMA = 20

def objective_sigma(trial):
    params = base_params_sigma | {
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'learning_rate': trial.suggest_float('lr', 0.03, 0.1, log=True),
        'min_child_samples': trial.suggest_int('min_child', 30, 200, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
    }
    cv = lgb.cv(params, dtrain_sigma, num_boost_round=1500, folds=folds,
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    return cv['valid l1-mean'][-1]

# Resume or start fresh
if on_drive('study_sigma.pkl'):
    load_drive('study_sigma.pkl')
    with open('study_sigma.pkl', 'rb') as f: study_sigma = pickle.load(f)
    done = len(study_sigma.trials)
    print(f'Resumed study_sigma: {done}/{TARGET_TRIALS_SIGMA} trials, best={study_sigma.best_value:.4f}')
else:
    study_sigma = optuna.create_study(
        direction='minimize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    )
    done = 0
    print('Starting fresh study_sigma')

remaining = max(0, TARGET_TRIALS_SIGMA - done)
if remaining > 0:
    print(f'Running {remaining} trials...'); sys.stdout.flush()
    t0 = time.time()
    study_sigma.optimize(objective_sigma, n_trials=remaining, show_progress_bar=True)
    print(f'Done in {(time.time()-t0)/60:.1f} min')
    with open('study_sigma.pkl', 'wb') as f: pickle.dump(study_sigma, f)
    save_drive('study_sigma.pkl')
else:
    print('All trials complete, loaded from Drive')

print(f'Best MAE={study_sigma.best_value:.4f}, params={study_sigma.best_params}')

In [ ]:
# ============================================================
# Cell 9: Train Final Model B (or load)
# ============================================================
if on_drive('model_sigma.txt'):
    load_drive('model_sigma.txt')
    model_sigma = lgb.Booster(model_file='model_sigma.txt')
    print('Loaded model_sigma from Drive')
else:
    bp = study_sigma.best_params
    best_p = base_params_sigma | {
        'num_leaves':        bp['num_leaves'],
        'learning_rate':     bp['lr'],
        'min_child_samples': bp['min_child'],
        'subsample':         bp['subsample'],
        'colsample_bytree':  bp['colsample'],
        'reg_alpha':         bp['reg_alpha'],
        'reg_lambda':        bp['reg_lambda'],
        'feature_pre_filter': False,
    }
    print(f'Best params: {best_p}')

    cv_s = lgb.cv(best_p, dtrain_sigma, num_boost_round=1500, folds=folds,
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    best_round_sigma = len(cv_s['valid l1-mean'])
    print(f'CV best round: {best_round_sigma}, CV MAE: {cv_s["valid l1-mean"][-1]:.6f}')

    model_sigma = lgb.train(best_p, dtrain_sigma, num_boost_round=best_round_sigma)
    model_sigma.save_model('model_sigma.txt')
    save_drive('model_sigma.txt')
    print(f'Trained model_sigma: {best_round_sigma} rounds')

sigma_pred_test = model_sigma.predict(X_test).clip(min=0.1)
resid_test = np.abs(y_test - mu_pred_test)
mae_sigma = mean_absolute_error(resid_test, sigma_pred_test)
print(f'Model B OOS MAE={mae_sigma:.4f}')

---
## Quantile Calibration

In [ ]:
# ============================================================
# Cell 10a: Calibration
# ============================================================
z10, z50, z90 = norm.ppf(0.10), 0.0, norm.ppf(0.90)
y_test_usd = np.exp(y_test) - 1

print('=== Calibration ===')
cal_results = []
for tau in [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]:
    z = norm.ppf(tau)
    q = np.exp(mu_pred_test + z * sigma_pred_test) - 1
    actual = (y_test_usd < q).mean()
    cal_results.append({'tau': tau, 'actual': float(actual), 'gap': float(actual - tau)})
    print(f'  tau={tau:.2f}: actual={actual:.3f} (gap={actual-tau:+.3f})')

q10_t = np.exp(mu_pred_test + z10 * sigma_pred_test) - 1
q50_t = np.exp(mu_pred_test) - 1
q90_t = np.exp(mu_pred_test + z90 * sigma_pred_test) - 1
mono = float(((q10_t <= q50_t) & (q50_t <= q90_t)).mean())
print(f'\nMonotonicity: {mono:.4f}')
print(f'Model A R2={r2:.4f}, Model B MAE={mae_sigma:.4f}')

In [ ]:
# ============================================================
# Cell 10b: Sigma calibration scaling
# ============================================================
from scipy.optimize import minimize_scalar

def calibration_loss(scale):
    """Find scale factor that minimizes calibration gap across quantiles."""
    sigma_scaled = sigma_pred_test * scale
    total_gap = 0
    for tau in [0.10, 0.25, 0.50, 0.75, 0.90]:
        z = norm.ppf(tau)
        q = np.exp(mu_pred_test + z * sigma_scaled) - 1
        actual = (y_test_usd < q).mean()
        total_gap += (actual - tau) ** 2
    return total_gap

result = minimize_scalar(calibration_loss, bounds=(1.0, 3.0), method='bounded')
SIGMA_SCALE = result.x
print(f'Optimal sigma scale factor: {SIGMA_SCALE:.3f}')

# Re-check calibration with scaled sigma
print('\n=== Calibration (after scaling) ===')
sigma_scaled_test = sigma_pred_test * SIGMA_SCALE
for tau in [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]:
    z = norm.ppf(tau)
    q = np.exp(mu_pred_test + z * sigma_scaled_test) - 1
    actual = (y_test_usd < q).mean()
    print(f'  tau={tau:.2f}: actual={actual:.3f} (gap={actual-tau:+.3f})')

---
## Counterfactual Matrix + DEA Clip

In [ ]:
# ============================================================
# Cell 11: Counterfactual Matrix (or load from Drive)
# ============================================================
if on_drive('env_model_output.npz'):
    load_drive('env_model_output.npz')
    d = np.load('env_model_output.npz', allow_pickle=True)
    q10_mat, q50_mat, q90_mat, sigma_mat = d['q10'], d['q50'], d['q90'], d['sigma']
    print(f'Loaded CF matrix from Drive: {q50_mat.shape}')
else:
    N_obs = len(df)
    q10_mat = np.full((N_obs, N_actions), np.nan, dtype=np.float32)
    q50_mat = np.full((N_obs, N_actions), np.nan, dtype=np.float32)
    q90_mat = np.full((N_obs, N_actions), np.nan, dtype=np.float32)
    sigma_mat = np.full((N_obs, N_actions), np.nan, dtype=np.float32)
    state_cols = [c for c in feature_cols if c not in action_features]

    print(f'Generating: {N_obs:,} x {N_actions}'); t0 = time.time()
    for aid in range(N_actions):
        crop_name = idx_to_crop[aid // 3]
        intensity = float(aid % 3)
        X_cf = df[state_cols].copy()
        X_cf['action_crop'] = pd.Categorical([crop_name]*N_obs,
                                              categories=df['action_crop'].cat.categories)
        X_cf['input_intensity'] = intensity
        X_cf = X_cf[feature_cols]
        mu_cf = model_mu.predict(X_cf)
        sig_cf = model_sigma.predict(X_cf).clip(min=0.1) * SIGMA_SCALE
        q10_mat[:, aid] = np.exp(mu_cf + z10 * sig_cf) - 1
        q50_mat[:, aid] = np.exp(mu_cf) - 1
        q90_mat[:, aid] = np.exp(mu_cf + z90 * sig_cf) - 1
        sigma_mat[:, aid] = (q90_mat[:, aid] - q10_mat[:, aid]) / 2.56
        if (aid+1) % 9 == 0: print(f'  {aid+1}/{N_actions}'); sys.stdout.flush()
    print(f'Done in {time.time()-t0:.1f}s')

    # DEA upper bound clip
    frontier = df['dea_frontier_value_USD'].values
    has_f = ~np.isnan(frontier)
    cap = frontier * 1.1
    for a in range(N_actions):
        m = has_f & (q90_mat[:, a] > cap)
        q90_mat[m, a] = cap[m]
    sigma_mat = np.where(np.isnan(q90_mat), np.nan,
                         np.clip(q90_mat - q10_mat, 0, None) / 2.56)
    print(f'DEA clip applied ({has_f.sum():,} obs with frontier)')

    # Save + push to Drive
    np.savez_compressed('env_model_output.npz',
        q10=q10_mat, q50=q50_mat, q90=q90_mat, sigma=sigma_mat,
        hh_id=df['hh_id_merge'].values, action_ids=np.arange(N_actions))
    save_drive('env_model_output.npz')
    print(f'Saved: {os.path.getsize("env_model_output.npz")/1e6:.1f} MB')

In [ ]:
# ============================================================
# Cell 11b: Counterfactual Matrix Diagnostics
# ============================================================
print('=== CF Matrix Diagnostics ===')
print(f'Shape: {q50_mat.shape}')

# 1. NaN check
nan_rate = np.isnan(q50_mat).mean()
print(f'NaN rate: {nan_rate:.4f}')

# 2. Monotonicity: q10 <= q50 <= q90
valid = ~np.isnan(q50_mat)
mono = ((q10_mat[valid] <= q50_mat[valid]) & (q50_mat[valid] <= q90_mat[valid])).mean()
print(f'Monotonicity (q10<=q50<=q90): {mono:.4f}')

# 3. Negative values
neg_q50 = (q50_mat[valid] < 0).mean()
neg_q10 = (q10_mat[valid] < 0).mean()
print(f'Negative q50: {neg_q50:.4f}, negative q10: {neg_q10:.4f}')

# 4. Value ranges (USD space)
print(f'q10: [{np.nanmin(q10_mat):.1f}, {np.nanmedian(q10_mat):.1f}, {np.nanmax(q10_mat):.1f}]')
print(f'q50: [{np.nanmin(q50_mat):.1f}, {np.nanmedian(q50_mat):.1f}, {np.nanmax(q50_mat):.1f}]')
print(f'q90: [{np.nanmin(q90_mat):.1f}, {np.nanmedian(q90_mat):.1f}, {np.nanmax(q90_mat):.1f}]')

# 5. Observed action rank analysis
obs_aid = df['action_id'].values
obs_q50 = q50_mat[np.arange(len(df)), obs_aid]
best_q50 = np.nanmax(q50_mat, axis=1)
worst_q50 = np.nanmin(q50_mat, axis=1)
chose_best = (np.abs(obs_q50 - best_q50) < 1.0).mean()
chose_worst = (np.abs(obs_q50 - worst_q50) < 1.0).mean()
print(f'Chose highest q50 action: {chose_best:.3f}')
print(f'Chose lowest q50 action: {chose_worst:.3f}')

# 6. Sigma spread
print(f'sigma: [{np.nanmin(sigma_mat):.2f}, {np.nanmedian(sigma_mat):.2f}, {np.nanmax(sigma_mat):.2f}]')

# 7. DEA clip impact
frontier = df['dea_frontier_value_USD'].values
has_f = ~np.isnan(frontier)
n_clipped = 0
for a in range(q90_mat.shape[1]):
    n_clipped += (has_f & (q90_mat[:, a] < (np.exp(model_mu.predict(X) + z90 * model_sigma.predict(X).clip(min=0.1) * SIGMA_SCALE) - 1)[:, None].squeeze() if False else False)).sum()
# Simplified: just count where q90 == q50 (fully clipped)
fully_clipped = (valid & (q90_mat == q50_mat)).sum()
print(f'Fully clipped (q90==q50): {fully_clipped:,} ({fully_clipped/valid.sum()*100:.2f}%)')

print(f'\n=== Summary ===')
print(f'{"✓" if mono == 1.0 else "✗"} Monotonicity = {mono:.4f}')
print(f'{"✓" if nan_rate < 0.5 else "✗"} NaN rate = {nan_rate:.4f}')
print(f'{"✓" if chose_best < 0.5 else "✗"} Chose best = {chose_best:.3f} (BIRL needs variation)')
print(f'{"✓" if neg_q50 < 0.05 else "⚠"} Negative q50 = {neg_q50:.4f}')

---
## Interpretability

In [ ]:
# ============================================================
# Cell 12: SHAP
# ============================================================
try:
    import shap
    explainer = shap.TreeExplainer(model_mu)
    shap_sample = X_test.sample(n=min(5000, len(X_test)), random_state=42)
    shap_values = explainer.shap_values(shap_sample)
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, shap_sample, feature_names=feature_cols, max_display=15, show=False)
    plt.tight_layout(); plt.savefig('shap_summary_mu.png', dpi=150)
    save_drive('shap_summary_mu.png')
    plt.show()
except Exception as e:
    print(f'SHAP error: {e}')

In [ ]:
# ============================================================
# Cell 13: Feature Importance
# ============================================================
imp_mu = pd.DataFrame({'feature': feature_cols,
    'importance': model_mu.feature_importance('gain')}).sort_values('importance', ascending=False)
imp_sigma = pd.DataFrame({'feature': feature_cols,
    'importance': model_sigma.feature_importance('gain')}).sort_values('importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(imp_mu['feature'][:15][::-1], imp_mu['importance'][:15][::-1])
axes[0].set_title('Model A (mu) Top 15')
axes[1].barh(imp_sigma['feature'][:15][::-1], imp_sigma['importance'][:15][::-1])
axes[1].set_title('Model B (sigma) Top 15')
plt.tight_layout(); plt.savefig('feature_importance.png', dpi=150)
save_drive('feature_importance.png')
plt.show()

In [ ]:
# ============================================================
# Cell 14: Calibration Plot
# ============================================================
cal_df = pd.DataFrame(cal_results)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0,1],[0,1],'k--',alpha=0.5,label='Perfect')
ax.scatter(cal_df['tau'], cal_df['actual'], s=80, zorder=5)
for _, r in cal_df.iterrows():
    ax.annotate(f"{r['tau']:.0%}", (r['tau'], r['actual']), textcoords='offset points', xytext=(8,0), fontsize=9)
ax.set_xlabel('Nominal quantile'); ax.set_ylabel('Actual coverage')
ax.set_title('Calibration: LogNormal quantiles')
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.legend()
plt.tight_layout(); plt.savefig('calibration_plot.png', dpi=150)
save_drive('calibration_plot.png')
plt.show()

# ============================================================
# Cell 14b: Calibration Plot (after sigma scaling)
# ============================================================
sigma_scaled_test = sigma_pred_test * SIGMA_SCALE
cal_scaled = []
for tau in [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]:
    z = norm.ppf(tau)
    q = np.exp(mu_pred_test + z * sigma_scaled_test) - 1
    actual = (y_test_usd < q).mean()
    cal_scaled.append({'tau': tau, 'actual': float(actual)})

cal_s_df = pd.DataFrame(cal_scaled)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0,1],[0,1],'k--',alpha=0.5,label='Perfect')
ax.scatter(cal_s_df['tau'], cal_s_df['actual'], s=80, zorder=5)
for _, r in cal_s_df.iterrows():
    ax.annotate(f"{r['tau']:.0%}", (r['tau'], r['actual']), textcoords='offset points', xytext=(8,0), fontsize=9)
ax.set_xlabel('Nominal quantile'); ax.set_ylabel('Actual coverage')
ax.set_title(f'Calibration: after σ recalibration (κ={SIGMA_SCALE:.3f})')
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.legend()
plt.tight_layout(); plt.savefig('calibration_scaled.png', dpi=150)
save_drive('calibration_scaled.png')
plt.show()
print(f'Max |gap| = {max(abs(r["actual"]-r["tau"]) for _,r in cal_s_df.iterrows()):.3f}')

---
## Save Everything to Drive (for local analysis)

In [ ]:
# ============================================================
# Cell 15: Save Full Results Package
# ============================================================

# 1. Metrics JSON
metrics = {
    'model_a_r2': float(r2),
    'model_a_rmse': float(rmse),
    'model_b_mae': float(mae_sigma),
    'monotonicity': float(mono),
    'sigma_scale': float(SIGMA_SCALE),
    'calibration_raw': cal_results,
    'calibration_scaled': [{'tau': r['tau'], 'actual': r['actual'], 'gap': r['actual'] - r['tau']}
                           for _, r in cal_s_df.iterrows()],
    'best_params_mu': {k: (float(v) if isinstance(v, (int, float, np.integer, np.floating)) else v)
                       for k, v in study_mu.best_params.items()},
    'best_params_sigma': {k: (float(v) if isinstance(v, (int, float, np.integer, np.floating)) else v)
                          for k, v in study_sigma.best_params.items()},
    'best_round_mu': int(model_mu.num_trees()),
    'n_obs': len(df),
    'n_obs_train': int(trainval_mask.sum()),
    'n_obs_test': int(test_mask.sum()),
    'n_actions': N_actions,
    'n_features': len(feature_cols),
    'feature_cols': feature_cols,
    'cat_features': cat_features,
}
with open('env_model_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
save_drive('env_model_metrics.json')

# 2. Feature importance CSVs
imp_mu.to_csv('importance_mu.csv', index=False); save_drive('importance_mu.csv')
imp_sigma.to_csv('importance_sigma.csv', index=False); save_drive('importance_sigma.csv')

# 3. OOS predictions (scaled sigma and quantiles)
sigma_scaled_oos = sigma_pred_test * SIGMA_SCALE
oos = pd.DataFrame({
    'hh_id_merge': df.loc[test_mask, 'hh_id_merge'].values,
    'country': df.loc[test_mask, 'country'].values,
    'action_crop': df.loc[test_mask, 'action_crop'].values,
    'action_id': df.loc[test_mask, 'action_id'].values,
    'y_true_log': y_test,
    'y_true_usd': y_test_usd,
    'mu_pred': mu_pred_test,
    'sigma_pred_raw': sigma_pred_test,
    'sigma_pred_scaled': sigma_scaled_oos,
    'q10_usd': np.exp(mu_pred_test + z10 * sigma_scaled_oos) - 1,
    'q50_usd': np.exp(mu_pred_test) - 1,
    'q90_usd': np.exp(mu_pred_test + z90 * sigma_scaled_oos) - 1,
    'residual': y_test - mu_pred_test,
    'abs_residual': np.abs(y_test - mu_pred_test),
})
oos.to_parquet('oos_predictions.parquet', index=False); save_drive('oos_predictions.parquet')

# 4. Train predictions
sigma_scaled_tv = model_sigma.predict(X_trainval).clip(min=0.1) * SIGMA_SCALE
train_pred = pd.DataFrame({
    'hh_id_merge': df.loc[trainval_mask, 'hh_id_merge'].values,
    'country': df.loc[trainval_mask, 'country'].values,
    'action_crop': df.loc[trainval_mask, 'action_crop'].values,
    'y_true_log': y_trainval,
    'mu_pred': mu_pred_tv,
    'sigma_pred_raw': model_sigma.predict(X_trainval).clip(min=0.1),
    'sigma_pred_scaled': sigma_scaled_tv,
    'residual': y_trainval - mu_pred_tv,
})
train_pred.to_parquet('train_predictions.parquet', index=False); save_drive('train_predictions.parquet')

# 5. Summary
print(f'\n{"="*50}')
print(f'All files saved to Drive: {DRIVE_DIR}')
print(f'{"="*50}')
for fname in sorted(os.listdir(DRIVE_DIR)):
    if fname.startswith('.'): continue
    sz = os.path.getsize(os.path.join(DRIVE_DIR, fname)) / 1e6
    print(f'  {fname:<40} {sz:>6.1f} MB')
print(f'\nKey metrics:')
print(f'  Model A R²={r2:.4f}, RMSE={rmse:.4f}')
print(f'  Model B MAE={mae_sigma:.4f}')
print(f'  Sigma scale={SIGMA_SCALE:.3f}')
print(f'  Monotonicity={mono:.4f}')
print(f'  Max calibration |gap| (scaled)={max(abs(r["actual"]-r["tau"]) for _,r in cal_s_df.iterrows()):.3f}')

In [ ]:
# ============================================================
# Cell 16: Summary
# ============================================================
print('='*60)
print('STEP 4 ENVIRONMENT MODEL — COMPLETE')
print('='*60)
print(f'Model A (mu):    R2={r2:.4f}, RMSE={rmse:.4f}')
print(f'Model B (sigma): MAE={mae_sigma:.4f}')
print(f'Monotonicity:    {mono:.4f}')
print(f'CF matrix:       {q50_mat.shape}')
print()
print('Files on Drive for LOCAL use (no retraining needed):')
print('  model_mu.txt / model_sigma.txt  -> lgb.Booster(model_file=...)')
print('  env_model_output.npz            -> np.load(...)["q10"/"q50"/"q90"/"sigma"]')
print('  oos_predictions.parquet          -> pd.read_parquet(...) for plotting')
print('  train_predictions.parquet        -> residual analysis')
print('  importance_mu/sigma.csv          -> feature importance bar charts')
print('  env_model_metrics.json           -> all metrics + hyperparams')
print('  study_mu/sigma.pkl               -> resume Optuna if needed')
print('  *.png                            -> figures ready for paper')